In [ ]:
# Assess accuracy of raw gridded products using all insitu stations that are valid
# Remove HAR and add terraclimate or aphrodite?
  
import xarray
import pandas as pd
import glob
import os
import rasterio
import concurrent.futures
from scipy.stats import spearmanr
import os
import numpy as np
from permetrics.regression import RegressionMetric
import HydroErr as he
import datetime as dt

In [ ]:
datasets = ['CHIRPS', 'PERSIANN', 'ERA5L', 'HAR', 'MSWEP'] 

#Read insitu data
df_insitu = pd.read_csv(r'..\Data\Processed\Insitu\insitu_rainfall_1991_2020.csv')
df_insitu['date'] = pd.to_datetime(df_insitu[['year', 'month', 'day']])

#Get path, files and file list for datasets
def extract_date(file, dataset):
    filename = os.path.basename(file)
    date_str = filename[-12:-4]
    return date_str

def get_dataset_info(dataset):
    dataset_path = fr"..\Data\Datasets\{dataset}_prec"
    dataset_tiff_files = glob.glob(os.path.join(dataset_path, "*.tif"))
    dataset_dates = list(map(lambda file: extract_date(file, dataset), dataset_tiff_files))
    return dataset_path, dataset_tiff_files, dataset_dates

dataset_results = {}
for dataset in datasets:
    dataset_path, dataset_tiff_files, dataset_dates = get_dataset_info(dataset)
    dataset_results[f'{dataset}_path'] = dataset_path
    dataset_results[f'{dataset}_files'] = dataset_tiff_files
    dataset_results[f'{dataset}_dates'] = dataset_dates

#dataset_results['MSWEP_files']

In [9]:
#extract station lat and lon in a list 
unique_stn = df_insitu[['index_no','lat','long']].drop_duplicates()
stn_points = list(zip(unique_stn['long'],unique_stn['lat']))
#stn_points
unique_stn.to_csv(r'../Data/unique_stn.csv',index=False)   

In [10]:
def extract_concurrent_grid_data(dataset_results, stn_points, unique_stn):
    df_final = None
    
    for dataset in datasets:
        files = dataset_results[f'{dataset}_files']
        print(f"Processing {dataset} with {len(files)} files...")
        
        def process_file(file):
            date_obj = extract_date(file, dataset) 
            local_records = []
            try:
                with rasterio.open(file) as src:
                    crs = src.crs
                    if crs == "EPSG:4326":
                        sampled_values = list(src.sample(stn_points))
                        for i, value in enumerate(sampled_values):
                            local_records.append({
                                'date': date_obj,
                                'stn_id': unique_stn.iloc[i]['index_no'],
                                'prec_grid': value[0]
                                })
            except Exception as e:
                print(f'Error processing {file}: {e}')
            return local_records

        dataset_records = []
        with concurrent.futures.ThreadPoolExecutor() as executor:
            results = executor.map(process_file, files)
            for result in results:
                dataset_records.extend(result)
        
        df_dataset = pd.DataFrame(dataset_records)
        
        # CHECK IF EMPTY FIRST
        if not df_dataset.empty:
            # df_dataset = df_dataset[df_dataset['prec_grid'] != -9999.0]
            # df_dataset = df_dataset.dropna().round(1)
            df_dataset.loc[df_dataset['prec_grid'] < 0, 'prec_grid'] = 0
            df_dataset = df_dataset.rename(columns={'prec_grid': f'{dataset}_prec'})
            
            # Merge logic
            if df_final is None:
                df_final = df_dataset
            else:
                df_final = pd.merge(df_final, df_dataset, on=['date', 'stn_id'], how='inner')
                print(f"Merged {dataset}. Resulting shape: {df_final.shape}")
        else:
            print(f"Warning: No data extracted for {dataset}")

    return df_final

df_grid_data = extract_concurrent_grid_data(dataset_results, stn_points, unique_stn)

Processing CHIRPS with 10957 files...
Processing PERSIANN with 10957 files...
Merged PERSIANN. Resulting shape: (720256, 4)
Processing ERA5L with 10957 files...
Merged ERA5L. Resulting shape: (720188, 5)
Processing HAR with 10957 files...
Merged HAR. Resulting shape: (720188, 6)
Processing MSWEP with 10958 files...
Merged MSWEP. Resulting shape: (720188, 7)


In [11]:
#merging station and grid df into a single df
df_insitu1 = df_insitu[['date','index_no','rainfall_sum']]
df_insitu1 = df_insitu1.rename(columns={'rainfall_sum':'prec_insitu', 'index_no':'stn_id'})

df_grid_data['date'] = pd.to_datetime(df_grid_data['date'])
df_insitu1['date'] = pd.to_datetime(df_insitu1['date'])
df_merged = pd.merge(df_grid_data, df_insitu1, on=['date','stn_id'], how='inner')

df_merged.to_csv(r'../Data/Processed/Accuracy_assessment/precipitation_daily_merged.csv', index=False)

**Calculating error metrices & CPI- Daily Scale**

In [12]:
#checkpoint - run from here to reduce code runtime while rerunning
import xarray
import pandas as pd
import glob
import os
import rasterio
import concurrent.futures
from scipy.stats import spearmanr
import os
import numpy as np
from permetrics.regression import RegressionMetric
import HydroErr as he
import datetime as dt

datasets = ['ERA5L', 'HAR', 'CHIRPS', 'MSWEP', 'PERSIANN']
df_merged = pd.read_csv(r'../Data/Processed/Accuracy_assessment/precipitation_daily_merged.csv')
df_merged = df_merged.round(1)
df_merged


,date,stn_id,CHIRPS_prec,PERSIANN_prec,ERA5L_prec,HAR_prec,MSWEP_prec,prec_insitu
0,1991-01-01,102.0,7.3,2.1,12.2,8.4,0.1,13.5
1,1991-01-01,103.0,7.9,0.0,13.8,4.6,0.1,5.2
2,1991-01-01,104.0,10.0,0.0,13.6,3.8,0.1,14.2
3,1991-01-01,105.0,7.2,0.0,11.2,1.3,0.0,16.4
4,1991-01-01,107.0,5.8,5.9,12.5,14.6,0.0,14.4
...,...,...,...,...,...,...,...,...
694213,2019-12-31,1407.0,0.0,0.0,0.1,0.0,0.0,0.0
694214,2019-12-31,1408.0,0.0,0.0,0.0,0.0,0.0,0.0
694215,2019-12-31,1412.0,0.0,0.0,0.0,0.0,0.0,0.0
694216,2019-12-31,1416.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
#column normalization function
def normalize_column(col):
    return (col - col.min()) / (col.max() - col.min())
          
#calculate accuracy of datasets
def calculate_accuracy(dataframe_prec):
    df_accuracy = pd.DataFrame()

    #datasets = ['ERA5L', 'HARv2', 'CHIRPS', 'MSWEP', 'PERSIANN']

    for dataset in datasets:
        dataset_accuracy = {'dataset': dataset}

        #Convert to array
        prec_grid_array = dataframe_prec[f'{dataset}_prec'].values
        prec_insitu_array = dataframe_prec['prec_insitu'].values
    
        #drop null values, and negative values
        valid_indices = np.logical_and.reduce([
            ~np.isnan(prec_grid_array),
            ~np.isnan(prec_insitu_array),
            prec_grid_array >=  0,
            prec_insitu_array >= 0
        ])
    
        observations = np.sum(valid_indices)
        dataset_accuracy['observations_n'] = observations

        prec_grid_array = prec_grid_array[valid_indices]
        prec_insitu_array = prec_insitu_array[valid_indices]

        #correlation coefficient
        correlation = he.spearman_r(prec_grid_array, prec_insitu_array)
        dataset_accuracy['correlation'] = correlation

        #NRMSE
        nrmse = he.nrmse_mean(prec_grid_array, prec_insitu_array)
        dataset_accuracy['nrmse'] = nrmse

        #Percent bias
        pbias = 100 * np.sum(prec_grid_array - prec_insitu_array) / np.sum(prec_insitu_array)
        dataset_accuracy['pbias'] = pbias

        #Relative standard deviation 
        rsd = (np.std(prec_grid_array))/(np.std(prec_insitu_array))
        dataset_accuracy['rsd'] = rsd

        df_dataset_accuracy = pd.DataFrame([dataset_accuracy])
        df_accuracy = pd.concat([df_accuracy, df_dataset_accuracy], ignore_index=True)

    #Normalization
    df_accuracy['abs_pbias'] = df_accuracy['pbias'].abs()
    df_accuracy['diff_rsd'] = (df_accuracy['rsd']-1).abs()

    df_accuracy_normalized = df_accuracy.copy().drop(columns=['pbias'])

    metrics = ['correlation', 'nrmse', 'abs_pbias', 'diff_rsd']
    for column in metrics:
        df_accuracy_normalized[column] = normalize_column(df_accuracy_normalized[column])

    #converting to scale of 0 to 1 (ideal to worst) - correlation & rsd needs inversion
    df_accuracy_normalized['correlation'] = 1 - df_accuracy_normalized['correlation']

    #get cpi for each (linear)
    df_accuracy_normalized['cpi'] = df_accuracy_normalized['correlation'] + df_accuracy_normalized['nrmse'] + df_accuracy_normalized['abs_pbias'] + df_accuracy_normalized['diff_rsd']
    df_accuracy_normalized['cpi_rank'] = df_accuracy_normalized['cpi'].rank(ascending=True)

    df_accuracy_final = df_accuracy_normalized[['dataset','cpi','cpi_rank']].merge(df_accuracy[['dataset','correlation','nrmse','pbias','rsd', 'observations_n']], on=['dataset'], how='inner')
    return df_accuracy_final

#get daily data accuracy
df_accuracy_daily = calculate_accuracy(df_merged)
df_accuracy_daily

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n
0,ERA5L,0.976071,1.0,0.578879,3.254415,35.659290,1.033788,687928
1,HAR,1.106481,2.0,0.532937,3.370347,16.443723,1.015424,687928
2,CHIRPS,2.067507,4.0,0.282060,3.424548,-11.323338,0.927618,687928
3,MSWEP,2.597588,5.0,0.432841,2.999038,-89.341380,0.140853,687928
4,PERSIANN,1.269158,3.0,0.418473,2.948812,-29.786414,0.569389,687928


**Accuracy and CPI Calculation - Seasonal Scale**

In [14]:
#define seasons
seasons = {
    'winter': [12, 1, 2],
    'pre_monsoon': [3, 4, 5],
    'monsoon': [6, 7, 8, 9],
    'post_monsoon': [10, 11]
}
df_merged1 = df_merged.copy()
df_merged1['date'] = pd.to_datetime(df_merged1['date'])
df_merged1['year'] = df_merged1['date'].dt.year
df_merged1['month'] = df_merged1['date'].dt.month

#create effective year for seasonal analysis
df_merged1['year_seasonal'] = df_merged1['year']
df_merged1.loc[df_merged1['month'] == 12, 'year_seasonal'] = df_merged1.loc[df_merged1['month'] == 12, 'year'] + 1

df_merged1['season'] = (df_merged1['month'].apply(lambda x: [k for k, v in seasons.items() if x in v][0]))

#calculate seasonal rainfall sum for each dataset 
df_seasonal = df_merged1.groupby(by = ['season','year_seasonal'])[['CHIRPS_prec',	'PERSIANN_prec', 'ERA5L_prec', 'prec_insitu','HAR_prec', 'MSWEP_prec']].sum().reset_index()
df_seasonal

,season,year_seasonal,CHIRPS_prec,PERSIANN_prec,ERA5L_prec,prec_insitu,HAR_prec,MSWEP_prec
0,monsoon,1991,63788.3,56566.8,95149.8,78883.7,81678.0,6844.8
1,monsoon,1992,49084.1,47566.9,74234.7,65389.0,63453.4,6301.5
2,monsoon,1993,66515.2,70279.6,113396.4,79830.4,83790.6,8389.2
3,monsoon,1994,65165.2,56914.0,92916.0,75957.2,76137.5,10469.5
4,monsoon,1995,64016.5,71442.4,123384.6,93145.5,100106.5,9547.2
...,...,...,...,...,...,...,...,...
112,winter,2016,1753.2,20184.6,5980.6,1547.3,3123.4,247.8
113,winter,2017,2155.8,1618.8,4769.9,1385.3,3552.5,204.0
114,winter,2018,2134.3,1337.7,3741.5,1146.9,2076.7,127.8
115,winter,2019,7024.3,6697.8,12707.7,7464.1,13817.8,801.6


In [15]:
#caluclate overall seasonal cpi
df_accuracy_seasonal = (calculate_accuracy(df_seasonal))
df_accuracy_seasonal['season'] = 'all'
#print("OVERALL SEASONAL ACCURACY:")
#print(df_accuracy_seasonal)

#calculate monsoonal cpi
df_monsoon = df_seasonal[df_seasonal['season'] == 'monsoon']
df_accuracy_monsoon = calculate_accuracy(df_monsoon)
df_accuracy_monsoon['season'] = 'monsoon'
#print("MONSOON ACCURACY:")
#print(df_accuracy_monsoon)

#calculate pre-monsoon cpi
df_premonsoon = df_seasonal[df_seasonal['season'] == 'pre_monsoon']
df_accuracy_premonsoon = calculate_accuracy(df_premonsoon)
df_accuracy_premonsoon['season'] = 'premonsoon'
#print("PRE-MONSOON ACCURACY:")
#print(df_accuracy_premonsoon)

#calculate post-monsoon cpi
df_postmonsoon = df_seasonal[df_seasonal['season'] == 'post_monsoon']
df_accuracy_postmonsoon = calculate_accuracy(df_postmonsoon)
df_accuracy_postmonsoon['season'] = 'postmonsoon'
#print("POST-MONSOON ACCURACY:")
#print(df_accuracy_postmonsoon)

#calculate winter cpi
df_winter = df_seasonal[df_seasonal['season'] == 'winter']
df_accuracy_winter = calculate_accuracy(df_winter)
df_accuracy_winter['season'] = 'winter'
#print("WINTER ACCURACY:")
#print(df_accuracy_winter)

#concatenate all cpi
df_accuracy_final = pd.concat([df_accuracy_seasonal, df_accuracy_monsoon, df_accuracy_premonsoon, df_accuracy_postmonsoon, df_accuracy_winter])
df_accuracy_final

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n,season
0,ERA5L,1.037215,3.0,0.979424,0.580601,36.994054,1.315060,117,all
1,HAR,0.117947,1.0,0.983021,0.282568,17.545504,1.073134,117,all
2,CHIRPS,1.031362,2.0,0.956705,0.246319,-10.591694,0.900923,117,all
3,MSWEP,3.770501,5.0,0.962745,1.473431,-89.248220,0.099659,117,all
4,PERSIANN,1.746771,4.0,0.959710,0.588997,-29.174131,0.641110,117,all
0,ERA5L,1.288197,4.0,0.641379,0.352189,33.047077,1.573244,29,monsoon
1,HAR,0.499530,1.0,0.664039,0.150210,9.342607,1.455509,29,monsoon
2,CHIRPS,0.911127,2.0,0.431527,0.148558,-10.177218,0.964526,29,monsoon
3,MSWEP,4.000000,5.0,0.405911,0.904681,-89.874043,0.119972,29,monsoon
4,PERSIANN,1.071956,3.0,0.550739,0.363336,-34.697559,0.935703,29,monsoon


**Accuracy and CPI Calculation - Annual Scale**

In [16]:
df_merged2 = df_merged.copy()
df_merged2['date'] = pd.to_datetime(df_merged2['date'])
df_merged2['year'] = df_merged2['date'].dt.year
df_annual = df_merged2.groupby(by = ['year'])[['CHIRPS_prec',	'PERSIANN_prec', 'ERA5L_prec', 'prec_insitu', 'HAR_prec', 'MSWEP_prec']].sum().reset_index()
df_accuracy_annual = calculate_accuracy(df_annual)
df_accuracy_annual

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n
0,ERA5L,1.200925,4.0,0.716749,0.382180,36.994054,1.429619,29
1,HAR,0.555967,1.0,0.750739,0.199377,17.545504,1.360691,29
2,CHIRPS,0.697682,3.0,0.454680,0.149307,-10.591694,1.043097,29
3,MSWEP,4.000000,5.0,0.300000,0.897898,-89.248220,0.110320,29
4,PERSIANN,0.675557,2.0,0.646305,0.304726,-29.174131,0.992959,29


**Accuracy and CPI Calculation - Monthly Scale**

In [17]:
df_merged3 = df_merged.copy()
df_merged3['date'] = pd.to_datetime(df_merged3['date'])
df_merged3['year'] = df_merged3['date'].dt.year
df_merged3['month'] = df_merged3['date'].dt.month
df_monthly = df_merged3.groupby(by = ['year','month'])[['CHIRPS_prec',	'PERSIANN_prec', 'ERA5L_prec', 'prec_insitu', 'HAR_prec', 'MSWEP_prec']].sum().reset_index()
df_accuracy_monthly = calculate_accuracy(df_monthly)
df_accuracy_monthly

,dataset,cpi,cpi_rank,correlation,nrmse,pbias,rsd,observations_n
0,ERA5L,0.938031,3.0,0.983167,0.584094,36.994054,1.336059,348
1,HAR,0.179926,1.0,0.983645,0.375285,17.545504,1.102435,348
2,CHIRPS,0.717598,2.0,0.968650,0.282906,-10.591694,0.901372,348
3,MSWEP,4.000000,5.0,0.962749,1.347642,-89.248220,0.100917,348
4,PERSIANN,1.594281,4.0,0.968714,0.592802,-29.174131,0.619266,348


**Calculation of Categorical Metrices for Dataset Evaluation**

In [18]:
import xarray as xr
import operator
import scores.categorical
df_merged

,date,stn_id,CHIRPS_prec,PERSIANN_prec,ERA5L_prec,HAR_prec,MSWEP_prec,prec_insitu
0,1991-01-01,102.0,7.3,2.1,12.2,8.4,0.1,13.5
1,1991-01-01,103.0,7.9,0.0,13.8,4.6,0.1,5.2
2,1991-01-01,104.0,10.0,0.0,13.6,3.8,0.1,14.2
3,1991-01-01,105.0,7.2,0.0,11.2,1.3,0.0,16.4
4,1991-01-01,107.0,5.8,5.9,12.5,14.6,0.0,14.4
...,...,...,...,...,...,...,...,...
694213,2019-12-31,1407.0,0.0,0.0,0.1,0.0,0.0,0.0
694214,2019-12-31,1408.0,0.0,0.0,0.0,0.0,0.0,0.0
694215,2019-12-31,1412.0,0.0,0.0,0.0,0.0,0.0,0.0
694216,2019-12-31,1416.0,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
#conversion to xarray
df_merged4 = df_merged.copy()

def calculate_categorical_metrics(dataframe_prec):
    df_categorical_metrics = pd.DataFrame()

    for dataset in datasets:
        dataset_categorical_metrics = {'dataset': dataset}

        obs_insitu = df_merged4.set_index(['stn_id', 'date'])['prec_insitu'].to_xarray()
        pred_dataset = df_merged4.set_index(['stn_id', 'date'])[f'{dataset}_prec'].to_xarray()

        #define event -  wet day is when precipitation is greater than 0.1
        event_operator = scores.categorical.ThresholdEventOperator(
            default_event_threshold=0.1, 
            default_op_fn=operator.ge)

        #conversion to binary
        dataset_binary, insitu_binary = event_operator.make_event_tables(pred_dataset, obs_insitu)
            
        #make contingency manager
        cm_dataset = event_operator.make_contingency_manager(dataset_binary, insitu_binary)
        
        #calculate metrics
        pod_dataset = cm_dataset.probability_of_detection()
        dataset_categorical_metrics['pod'] = pod_dataset.values
        far_dataset = cm_dataset.false_alarm_rate()
        dataset_categorical_metrics['far'] = far_dataset.values
        fb_dataset = cm_dataset.frequency_bias() 
        dataset_categorical_metrics['fb'] = fb_dataset.values
        csi_dataset = cm_dataset.critical_success_index()
        dataset_categorical_metrics['csi'] = csi_dataset.values
        
        # Wrap in a list to create a single-row DataFrame
        dataset_categorical_metrics = pd.DataFrame([dataset_categorical_metrics])
        df_categorical_metrics = pd.concat([df_categorical_metrics, dataset_categorical_metrics], ignore_index=True)

    return df_categorical_metrics

df_categorical_metrics = calculate_categorical_metrics(df_merged4)
numeric_cols = ['pod', 'far', 'fb', 'csi']
df_categorical_metrics[numeric_cols] = df_categorical_metrics[numeric_cols].astype(float).round(2)
df_categorical_metrics

,dataset,pod,far,fb,csi
0,ERA5L,0.98,0.67,2.68,0.36
1,HAR,0.85,0.34,1.72,0.46
2,CHIRPS,0.33,0.12,0.63,0.26
3,MSWEP,0.67,0.24,1.28,0.41
4,PERSIANN,0.77,0.37,1.69,0.40
